In [1]:
import requests
import pandas as pd
import time

coins = ["bitcoin", "ethereum", "binancecoin", "solana", "ripple"]
labels = ["BTC", "ETH", "BNB", "SOL", "XRP"]

all_data = {}

for coin, label in zip(coins, labels):
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart"
    params = {"vs_currency": "usd", "days": "365", "interval": "daily"}
    response = requests.get(url, params=params)
    prices = response.json()["prices"]
    df_temp = pd.DataFrame(prices, columns=["timestamp", label])
    df_temp["Date"] = pd.to_datetime(df_temp["timestamp"], unit="ms").dt.date
    df_temp = df_temp[["Date", label]]
    all_data[label] = df_temp
    time.sleep(12)  # Respect rate limit CoinGecko (30 calls/min)
    print(f"✅ {label} recovered")

print("✅ All recovered assets")

✅ BTC recovered
✅ ETH recovered
✅ BNB recovered
✅ SOL recovered
✅ XRP recovered
✅ All recovered assets


In [3]:
from functools import reduce

# Merge all assets on the date
df_all = reduce(lambda a, b: pd.merge(a, b, on="Date", how="inner"), all_data.values())
df_all = df_all.sort_values("Date").reset_index(drop=True)

# Daily yields
for label in labels:
    df_all[f"{label}_return"] = df_all[label].pct_change()

# Performance metrics (annualized)
results = []
for label in labels:
    col = f"{label}_return"
    annual_yield = df_all[col].mean() * 365
    volatility        = df_all[col].std() * (365 ** 0.5)
    sharpe            = annual_yield / volatility if volatility != 0 else 0
    max_price         = df_all[label].max()
    min_price         = df_all[label].min()
    
    results.append({
        "Coin":              label,
        "Annual_Yield":  round(annual_yield * 100, 2),
        "Volatility":        round(volatility * 100, 2),
        "Sharpe_Ratio":      round(sharpe, 3),
        "Max_Price_USD":      round(max_price, 2),
        "Min_Price_USD":      round(min_price, 2)
    })

df_metrics = pd.DataFrame(results).sort_values("Sharpe_Ratio", ascending=False)
print(df_metrics)

  Coin  Annual_Yield  Volatility  Sharpe_Ratio  Max_Price_USD  Min_Price_USD
1  ETH         36.73       72.63         0.506        4829.23        1471.36
2  BNB         13.03       50.22         0.259        1311.71         553.40
3  SOL        -11.65       73.06        -0.159         247.56          77.74
4  XRP        -18.76       69.28        -0.271           3.56           1.22
0  BTC        -11.64       42.47        -0.274      124773.51       62853.69


In [4]:
# Export
df_all.to_csv("historical_crypto_prices.csv", index=False)
df_metrics.to_csv("crypto_metrics.csv", index=False)
print("✅ Successful exports")

✅ Successful exports


In [1]:
import pandas as pd

# Load both datasets
df_metrics = pd.read_csv("crypto_metrics.csv")
df_prices = pd.read_csv("historical_crypto_prices.csv")

# Display variable types
print("=== crypto_metrics.csv ===")
print(df_metrics.dtypes)
print("\n=== historical_crypto_prices.csv ===")
print(df_prices.dtypes)

=== crypto_metrics.csv ===
Coin              object
Annual_Yield     float64
Volatility       float64
Sharpe_Ratio     float64
Max_Price_USD    float64
Min_Price_USD    float64
dtype: object

=== historical_crypto_prices.csv ===
Date           object
BTC           float64
ETH           float64
BNB           float64
SOL           float64
XRP           float64
BTC_return    float64
ETH_return    float64
BNB_return    float64
SOL_return    float64
XRP_return    float64
dtype: object


In [2]:
# Check missing values count and percentage
print("=== crypto_metrics.csv — Missing Values ===")
print(df_metrics.isnull().sum())
print(df_metrics.isnull().mean().mul(100).round(2).astype(str) + " %")

print("\n=== historical_crypto_prices.csv — Missing Values ===")
print(df_prices.isnull().sum())
print(df_prices.isnull().mean().mul(100).round(2).astype(str) + " %")

=== crypto_metrics.csv — Missing Values ===
Coin             0
Annual_Yield     0
Volatility       0
Sharpe_Ratio     0
Max_Price_USD    0
Min_Price_USD    0
dtype: int64
Coin             0.0 %
Annual_Yield     0.0 %
Volatility       0.0 %
Sharpe_Ratio     0.0 %
Max_Price_USD    0.0 %
Min_Price_USD    0.0 %
dtype: object

=== historical_crypto_prices.csv — Missing Values ===
Date          0
BTC           0
ETH           0
BNB           0
SOL           0
XRP           0
BTC_return    1
ETH_return    1
BNB_return    1
SOL_return    1
XRP_return    1
dtype: int64
Date           0.0 %
BTC            0.0 %
ETH            0.0 %
BNB            0.0 %
SOL            0.0 %
XRP            0.0 %
BTC_return    0.25 %
ETH_return    0.25 %
BNB_return    0.25 %
SOL_return    0.25 %
XRP_return    0.25 %
dtype: object


In [3]:
# IQR outlier detection on crypto_metrics numeric columns
num_cols_metrics = ["Annual_Yield", "Volatility", "Sharpe_Ratio", "Max_Price_USD", "Min_Price_USD"]

print("=== crypto_metrics.csv — Outliers (IQR) ===")
for col in num_cols_metrics:
    Q1 = df_metrics[col].quantile(0.25)
    Q3 = df_metrics[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_metrics[(df_metrics[col] < lower) | (df_metrics[col] > upper)]
    print(f"{col}: {len(outliers)} outlier(s) | Bounds [{lower:.2f}, {upper:.2f}]")

# IQR outlier detection on price columns in historical prices
price_cols = ["BTC", "ETH", "BNB", "SOL", "XRP"]

print("\n=== historical_crypto_prices.csv — Outliers on Prices (IQR) ===")
for col in price_cols:
    Q1 = df_prices[col].quantile(0.25)
    Q3 = df_prices[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_prices[(df_prices[col] < lower) | (df_prices[col] > upper)]
    print(f"{col}: {len(outliers)} outlier(s) | Bounds [{lower:.2f}, {upper:.2f}]")

=== crypto_metrics.csv — Outliers (IQR) ===
Annual_Yield: 0 outlier(s) | Bounds [-48.67, 50.05]
Volatility: 0 outlier(s) | Bounds [16.61, 106.24]
Sharpe_Ratio: 0 outlier(s) | Bounds [-1.07, 1.05]
Max_Price_USD: 1 outlier(s) | Bounds [-6624.94, 11701.73]
Min_Price_USD: 1 outlier(s) | Bounds [-2012.69, 3561.79]

=== historical_crypto_prices.csv — Outliers on Prices (IQR) ===
BTC: 0 outlier(s) | Bounds [43554.43, 150227.25]
ETH: 0 outlier(s) | Bounds [-197.25, 6037.72]
BNB: 4 outlier(s) | Bounds [234.50, 1284.55]
SOL: 0 outlier(s) | Bounds [31.19, 264.51]
XRP: 0 outlier(s) | Bounds [0.80, 3.60]


In [4]:
# CORRECTION 1 — Drop the first row of df_prices (missing return values, no prior date)
df_prices = df_prices.dropna(subset=["BTC_return", "ETH_return", "BNB_return", "SOL_return", "XRP_return"])

# Verify correction
print("=== After correction — Missing values in return columns ===")
print(df_prices[["BTC_return", "ETH_return", "BNB_return", "SOL_return", "XRP_return"]].isnull().sum())
print(f"Remaining rows: {len(df_prices)}")

# CORRECTION 2 — No outlier removal: extreme prices are real market events
# Keeping them preserves the integrity of yield/risk analysis
print("\n=== No outlier removal applied — extreme prices are real market data ===")

=== After correction — Missing values in return columns ===
BTC_return    0
ETH_return    0
BNB_return    0
SOL_return    0
XRP_return    0
dtype: int64
Remaining rows: 395

=== No outlier removal applied — extreme prices are real market data ===


In [5]:
# Load datasets
df_metrics = pd.read_csv("crypto_metrics.csv")
df_prices = pd.read_csv("historical_crypto_prices.csv")

# Drop first row (missing returns — structural, fixed in Step 1)
df_prices = df_prices.dropna(subset=["BTC_return","ETH_return","BNB_return","SOL_return","XRP_return"])

# ── CONVERT ──────────────────────────────────────────────────
# Convert Date from object to datetime → required by SQL DATE and Power BI time axis
df_prices["Date"] = pd.to_datetime(df_prices["Date"])

# ── NORMALIZE ────────────────────────────────────────────────
# Prices have very different scales (BTC ~80k vs XRP ~2)
# Min-Max normalization enables cross-coin visual comparison in Power BI
price_cols = ["BTC", "ETH", "BNB", "SOL", "XRP"]

for col in price_cols:
    min_val = df_prices[col].min()
    max_val = df_prices[col].max()
    # Normalized price: 0 = historical low, 1 = historical high
    df_prices[col + "_normalized"] = (df_prices[col] - min_val) / (max_val - min_val)

# ── VERIFY ───────────────────────────────────────────────────
print("=== df_prices dtypes after conversion ===")
print(df_prices.dtypes)

print("\n=== Normalized price range check (should be 0 to 1) ===")
norm_cols = [c + "_normalized" for c in price_cols]
print(df_prices[norm_cols].agg(["min", "max"]))

print("\n=== df_metrics dtypes (no conversion needed) ===")
print(df_metrics.dtypes)

=== df_prices dtypes after conversion ===
Date              datetime64[ns]
BTC                      float64
ETH                      float64
BNB                      float64
SOL                      float64
XRP                      float64
BTC_return               float64
ETH_return               float64
BNB_return               float64
SOL_return               float64
XRP_return               float64
BTC_normalized           float64
ETH_normalized           float64
BNB_normalized           float64
SOL_normalized           float64
XRP_normalized           float64
dtype: object

=== Normalized price range check (should be 0 to 1) ===
     BTC_normalized  ETH_normalized  BNB_normalized  SOL_normalized  \
min             0.0             0.0             0.0             0.0   
max             1.0             1.0             1.0             1.0   

     XRP_normalized  
min             0.0  
max             1.0  

=== df_metrics dtypes (no conversion needed) ===
Coin              object
Annu

In [6]:
# ── crypto_metrics.csv — Rename columns ──────────────────────
df_metrics = df_metrics.rename(columns={
    "Coin"         : "crypto_name",
    "Annual_Yield" : "annual_return_pct",
    "Volatility"   : "risk_level_pct",
    "Sharpe_Ratio" : "return_risk_ratio",
    "Max_Price_USD": "price_max_usd",
    "Min_Price_USD": "price_min_usd"
})

# ── historical_crypto_prices.csv — Rename columns ────────────
df_prices = df_prices.rename(columns={
    "BTC"            : "btc_price_usd",
    "ETH"            : "eth_price_usd",
    "BNB"            : "bnb_price_usd",
    "SOL"            : "sol_price_usd",
    "XRP"            : "xrp_price_usd",
    "BTC_return"     : "btc_daily_return",
    "ETH_return"     : "eth_daily_return",
    "BNB_return"     : "bnb_daily_return",
    "SOL_return"     : "sol_daily_return",
    "XRP_return"     : "xrp_daily_return",
    "BTC_normalized" : "btc_price_indexed",
    "ETH_normalized" : "eth_price_indexed",
    "BNB_normalized" : "bnb_price_indexed",
    "SOL_normalized" : "sol_price_indexed",
    "XRP_normalized" : "xrp_price_indexed"
})

# ── VERIFY ───────────────────────────────────────────────────
print("=== df_metrics — New column names ===")
print(df_metrics.columns.tolist())

print("\n=== df_prices — New column names ===")
print(df_prices.columns.tolist())

=== df_metrics — New column names ===
['crypto_name', 'annual_return_pct', 'risk_level_pct', 'return_risk_ratio', 'price_max_usd', 'price_min_usd']

=== df_prices — New column names ===
['Date', 'btc_price_usd', 'eth_price_usd', 'bnb_price_usd', 'sol_price_usd', 'xrp_price_usd', 'btc_daily_return', 'eth_daily_return', 'bnb_daily_return', 'sol_daily_return', 'xrp_daily_return', 'btc_price_indexed', 'eth_price_indexed', 'bnb_price_indexed', 'sol_price_indexed', 'xrp_price_indexed']


In [7]:
# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEERING — crypto_metrics
# ══════════════════════════════════════════════════════════════

# risk_category: segment volatility into readable risk levels
# Thresholds: <50% = Low, 50-65% = Medium, >65% = High
df_metrics["risk_category"] = pd.cut(
    df_metrics["risk_level_pct"],
    bins    = [0, 50, 65, float("inf")],
    labels  = ["Low Risk", "Medium Risk", "High Risk"]
)

# performance_label: flag whether return/risk ratio is efficient
# Positive Sharpe = coin compensates its risk / Negative = it does not
df_metrics["performance_label"] = df_metrics["return_risk_ratio"].apply(
    lambda x: "Efficient" if x >= 0 else "Inefficient"
)

# price_range_usd: total price amplitude over the period
# Measures how wide the trading range was for each coin
df_metrics["price_range_usd"] = df_metrics["price_max_usd"] - df_metrics["price_min_usd"]

# ── VERIFY metrics ────────────────────────────────────────────
print("=== df_metrics — New features ===")
print(df_metrics[["crypto_name","risk_category","performance_label","price_range_usd"]])


# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEERING — historical_crypto_prices
# ══════════════════════════════════════════════════════════════

# month: extract month number for monthly trend analysis in Power BI
df_prices["month"] = df_prices["Date"].dt.month

# quarter: extract quarter label for Q1/Q2/Q3/Q4 segmentation
df_prices["quarter"] = df_prices["Date"].dt.to_period("Q").astype(str)

# year: extract year for annual slicer filter
df_prices["year"] = df_prices["Date"].dt.year

# return_direction columns: classify each daily return as Up / Down / Stable
# Enables counting bullish vs bearish days per coin
return_cols = {
    "btc_daily_return" : "btc_return_direction",
    "eth_daily_return" : "eth_return_direction",
    "bnb_daily_return" : "bnb_return_direction",
    "sol_daily_return" : "sol_return_direction",
    "xrp_daily_return" : "xrp_return_direction"
}

for source_col, new_col in return_cols.items():
    df_prices[new_col] = df_prices[source_col].apply(
        lambda x: "Up" if x > 0 else ("Down" if x < 0 else "Stable")
    )

# ── VERIFY prices ─────────────────────────────────────────────
print("\n=== df_prices — New temporal features ===")
print(df_prices[["Date","month","quarter","year"]].head())

print("\n=== df_prices — Return direction sample ===")
direction_cols = list(return_cols.values())
print(df_prices[direction_cols].head(10))

print("\n=== Return direction value counts — BTC ===")
print(df_prices["btc_return_direction"].value_counts())

=== df_metrics — New features ===
  crypto_name risk_category performance_label  price_range_usd
0         ETH     High Risk         Efficient          3357.87
1         BNB   Medium Risk         Efficient           758.31
2         SOL     High Risk       Inefficient           169.82
3         XRP     High Risk       Inefficient             2.34
4         BTC      Low Risk       Inefficient         61919.82

=== df_prices — New temporal features ===
        Date  month quarter  year
1 2025-04-03      4  2025Q2  2025
2 2025-04-04      4  2025Q2  2025
3 2025-04-05      4  2025Q2  2025
4 2025-04-06      4  2025Q2  2025
5 2025-04-07      4  2025Q2  2025

=== df_prices — Return direction sample ===
   btc_return_direction eth_return_direction bnb_return_direction  \
1                  Down                 Down                 Down   
2                    Up                   Up                   Up   
3                    Up                 Down                   Up   
4                  D

In [8]:
# ══════════════════════════════════════════════════════════════
# VARIABLE SELECTION — Drop intermediate price columns
# price_max_usd and price_min_usd are captured in price_range_usd
# Dropped only from the Power BI export — SQL keeps full dataset
# ══════════════════════════════════════════════════════════════

# Drop redundant columns from df_metrics
df_metrics = df_metrics.drop(columns=["price_max_usd", "price_min_usd"])

# ── VERIFY ────────────────────────────────────────────────────
print("=== df_metrics — Final columns ===")
print(df_metrics.columns.tolist())
print(f"Shape: {df_metrics.shape}")

print("\n=== df_metrics — Final dataset preview ===")
print(df_metrics)

print("\n=== df_prices — Final columns ===")
print(df_prices.columns.tolist())
print(f"Shape: {df_prices.shape}")

# ══════════════════════════════════════════════════════════════
# EXPORT — Save final cleaned datasets
# ══════════════════════════════════════════════════════════════

# Export df_metrics to CSV for SQL import
df_metrics.to_csv("crypto_metrics_clean.csv", index=False)

# Export df_prices to CSV for SQL import
df_prices.to_csv("historical_crypto_prices_clean.csv", index=False)

print("\n=== Export completed ===")
print("crypto_metrics_clean.csv        → ready for MySQL import")
print("historical_crypto_prices_clean.csv → ready for MySQL import")

=== df_metrics — Final columns ===
['crypto_name', 'annual_return_pct', 'risk_level_pct', 'return_risk_ratio', 'risk_category', 'performance_label', 'price_range_usd']
Shape: (5, 7)

=== df_metrics — Final dataset preview ===
  crypto_name  annual_return_pct  risk_level_pct  return_risk_ratio  \
0         ETH              36.73           72.63              0.506   
1         BNB              13.03           50.22              0.259   
2         SOL             -11.65           73.06             -0.159   
3         XRP             -18.76           69.28             -0.271   
4         BTC             -11.64           42.47             -0.274   

  risk_category performance_label  price_range_usd  
0     High Risk         Efficient          3357.87  
1   Medium Risk         Efficient           758.31  
2     High Risk       Inefficient           169.82  
3     High Risk       Inefficient             2.34  
4      Low Risk       Inefficient         61919.82  

=== df_prices — Final column

In [10]:
df_metrics.to_excel("crypto_metrics_clean.xlsx",index=False,sheet_name="crypto_metrics")
df_prices.to_excel("historical_crypto_prices_clean.xlsx",index=False,sheet_name="historical_crypto_prices")

In [11]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [12]:
engine.connect()

In [13]:
df_metrics.to_sql(
    name="crypto_metrics_clean",
    con=engine,
    if_exists="replace",
    index=False
)
df_prices.to_sql(
    name="historical_crypto_prices_clean",
    con=engine,
    if_exists="replace",
    index=False
)

395